In [295]:
import os
import torch
from torch import nn
from torch.utils.data import dataloader
from torchvision import datasets, transforms

In [296]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [297]:
class NeuralNetwrok(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,10)
        )
    def forward(self, x):
            x =self.flatten(x)
            logits = self.linear_relu_stack(x)
            return logits



In [298]:
model  = NeuralNetwrok().to(device)
print(model)

NeuralNetwrok(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [299]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
logits

tensor([[-0.0045, -0.0780,  0.0125,  0.0258,  0.0357, -0.0010, -0.2036,  0.0230,
         -0.0395, -0.0097]], device='mps:0', grad_fn=<LinearBackward0>)

In [300]:
pred_probab = nn.Softmax(dim=1)(logits)
pred_probab[0]

tensor([0.1017, 0.0945, 0.1035, 0.1049, 0.1059, 0.1021, 0.0834, 0.1046, 0.0982,
        0.1012], device='mps:0', grad_fn=<SelectBackward0>)

In [301]:
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([4], device='mps:0')


In [302]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


In [303]:
flatten = nn.Flatten()
flatten_image = flatten(input_image)
flatten_image[0].size()

torch.Size([784])

In [304]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flatten_image)
print(hidden1.size())

torch.Size([3, 20])


In [305]:
print(f"Before ReLU : {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU : tensor([[ 0.5540, -1.0423, -0.3714,  0.6234,  0.5323,  0.4645, -0.3341,  0.6385,
          0.2528, -0.0478,  0.0272, -0.4183, -0.1224,  0.1611,  0.6004, -0.2434,
          0.0988, -0.0526,  0.4503, -0.5371],
        [ 0.0209, -0.8032, -0.2086,  0.6046, -0.1793,  0.3963, -0.4888,  0.7045,
          0.1883, -0.0888,  0.2431, -0.2239, -0.1453, -0.1837,  0.3531,  0.1501,
          0.2070,  0.3517,  0.4665, -0.4909],
        [ 0.2191, -0.3969, -0.3290,  0.2469,  0.3247,  0.2123, -0.3777,  0.1188,
          0.2818, -0.2782,  0.2666, -0.1640,  0.2572,  0.2070,  0.4795, -0.1393,
         -0.2247,  0.3412,  0.5935, -0.1237]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.5540, 0.0000, 0.0000, 0.6234, 0.5323, 0.4645, 0.0000, 0.6385, 0.2528,
         0.0000, 0.0272, 0.0000, 0.0000, 0.1611, 0.6004, 0.0000, 0.0988, 0.0000,
         0.4503, 0.0000],
        [0.0209, 0.0000, 0.0000, 0.6046, 0.0000, 0.3963, 0.0000, 0.7045, 0.1883,
         0.0000, 0.2431, 0.0000, 0.0000, 0.0000, 0.3

In [306]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20,10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)
logits

tensor([[ 0.0608,  0.1667, -0.2128,  0.1339, -0.0920, -0.0226,  0.1592,  0.1130,
         -0.1517,  0.4360],
        [ 0.2104,  0.2978, -0.4159,  0.1446, -0.0969,  0.0715,  0.0987, -0.0132,
         -0.3080,  0.6304],
        [ 0.1460,  0.2478, -0.3739,  0.1361, -0.0859,  0.0065,  0.1152,  0.0668,
         -0.2166,  0.5063]], grad_fn=<AddmmBackward0>)

In [307]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)
pred_probab

tensor([[0.0985, 0.1096, 0.0750, 0.1060, 0.0846, 0.0907, 0.1087, 0.1038, 0.0797,
         0.1434],
        [0.1114, 0.1215, 0.0595, 0.1043, 0.0819, 0.0969, 0.0996, 0.0891, 0.0663,
         0.1695],
        [0.1066, 0.1181, 0.0634, 0.1056, 0.0846, 0.0927, 0.1034, 0.0985, 0.0742,
         0.1529]], grad_fn=<SoftmaxBackward0>)

In [308]:
print(f"Model Structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values: {param[:2]}\n")

Model Structure: NeuralNetwrok(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values: tensor([[-0.0198, -0.0272,  0.0335,  ...,  0.0232,  0.0243, -0.0288],
        [ 0.0002,  0.0140, -0.0115,  ..., -0.0080, -0.0206, -0.0303]],
       device='mps:0', grad_fn=<SliceBackward0>)

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values: tensor([-0.0327, -0.0348], device='mps:0', grad_fn=<SliceBackward0>)

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values: tensor([[ 0.0345,  0.0016, -0.0113,  ...,  0.0040, -0.0414, -0.0170],
        [ 0.0268, -0.0172,  0.0220,  ...,  0.0157,  0.0200, -0.0407]],
       device='mps:0', grad_fn=<SliceBackw